In [ ]:
import os
if 'experiments' in os.getcwd (): os.chdir (os.getcwd () + "/..")

####### DATA PATHS #######
TRAIN_PATH = "./data/in/cattle_data_train.csv"
TEST_PATH = "./data/in/cattle_data_test.csv"
OUT_PATH = "./data/out/nnet_fold_ensemble.csv"

######## MOOOOO ##########
SEED = int ("".join (str (ord (c)) for c in 
            """  __________________
               < GORDOBOB GOT MILK! >
                 ------------------
                         \   ^__^
                          \  (oo)\________
                             (__)\        )\ 
                                 ||----w-||
                                 ||    ; ||
                      so much milk -> """)) % (2 ** 32)

<>:11: SyntaxWarning: invalid escape sequence '\ '
<>:11: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_263558/3646175837.py:11: SyntaxWarning: invalid escape sequence '\ '
  """  __________________


# Preprocessing

In [2]:
TARGET_FEATURE =            "Milk_Yield_L"

DROP_FEATURES =            ["Cattle_ID",
                            "Farm_ID",
                            "Feed_Quantity_lb",
                            "Climate_Zone",
                            "Management_System",
                            "Feed_Type",
                            "Feeding_Frequency",
                            "Walking_Distance_km",
                            "Grazing_Duration_hrs",
                            "Rumination_Time_hrs",
                            "Resting_Hours",
                            "Humidity_percent",
                            "BVD_Vaccine",
                            "FMD_Vaccine",
                            "Brucellosis_Vaccine",
                            "HS_Vaccine",
                            "BQ_Vaccine",
                            "Housing_Score",
                            "Body_Condition_Score",
                            "Milking_Interval_hrs",
                            "Breed",
                            "Date"]

CATEGORICAL_FEATURES =     ["Season",
                            "Young",
                            "Lactation_Stage",
                            "IBR_Vaccine",
                            "Anthrax_Vaccine",
                            "Rabies_Vaccine"]

STANDARD_SCALED_FEATURES = ["Feed_Quantity_kg",
                            "Water_Intake_L",
                            "Parity",
                            "Ambient_Temperature_C",
                            "Previous_Week_Avg_Yield",
                            "Days_in_Milk",
                            "Age_Months",
                            "Weight_kg"]

In [3]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

def preprocess (
    dtrain, dtest, scaler = None
) -> tuple[pd.DataFrame, pd.DataFrame, StandardScaler]:
    """
    NOTES:
    - Interaction features do not help
    - Squaring feed for outlier overexageratting does not help
    - clipping negative records worsens results
    - Predictive and mean imputation worsen results
    - Robust & min-max scalers worsen results
    """
    ### CONVERT MONTH TO SEASON ###
    def month_to_season (m):
        if m in [12, 1, 2]:
            return "Winter"
        elif m in [3, 4, 5]:
            return "Spring"
        elif m in [6, 7, 8]:
            return "Summer"
        else:
            return "Fall"

    months = pd.to_datetime (dtest['Date']).dt.month
    dtest['Season'] = months.apply (month_to_season)

    months = pd.to_datetime (dtrain['Date']).dt.month
    dtrain['Season'] = months.apply (month_to_season)

    ### EXTRACT AGE/YOUTH PATTERN ###
    dtrain['Young'] = (dtrain['Age_Months'] < 60).astype (int)
    dtest['Young'] = (dtest['Age_Months'] < 60).astype (int)

    ### IMPUT MISSING FEED ###
    median_val = dtrain["Feed_Quantity_kg"].median ()

    dtrain.loc[dtrain["Feed_Quantity_kg"].isna (), "Feed_Quantity_kg"] = median_val
    dtest.loc[dtest["Feed_Quantity_kg"].isna (), "Feed_Quantity_kg"] = median_val

    ### ONE-HOT ENCODE CATEGORICALS ###
    dtrain = pd.get_dummies (dtrain, columns = CATEGORICAL_FEATURES, 
                             drop_first = True)
    dtest = pd.get_dummies (dtest, columns = CATEGORICAL_FEATURES, 
                            drop_first = True)
    dtrain, dtest = dtrain.align (dtest, join = 'left', axis = 1, fill_value = 0)

    ### STANDARDIZE DATA ###
    if scaler is None:
        scaler = StandardScaler ()
        dtrain[STANDARD_SCALED_FEATURES] = scaler.fit_transform (dtrain[STANDARD_SCALED_FEATURES])
    else:
        dtrain[STANDARD_SCALED_FEATURES] = scaler.transform (dtrain[STANDARD_SCALED_FEATURES])
    
    dtest[STANDARD_SCALED_FEATURES] = scaler.transform (dtest[STANDARD_SCALED_FEATURES])
    
    ### DROP ALL THE USELESS FEATURES ###
    dtrain = dtrain.drop (DROP_FEATURES, axis = 1)
    dtest = dtest.drop (DROP_FEATURES, axis = 1)

    ### DONE >;D ###
    return dtrain, dtest, scaler

# Training Set Eval

In [4]:
from sklearn.neural_network import MLPRegressor

total_iterations = 0
iterations_per_step = 10
model_template = MLPRegressor (hidden_layer_sizes = (110, 110, 110),
                               activation = "tanh",
                               learning_rate_init = 0.00003,
                               learning_rate = "adaptive",
                               early_stopping = False, 
                               n_iter_no_change = 20,
                               verbose = False,
                               warm_start = True,
                               max_iter = iterations_per_step,
                               random_state = SEED)

In [5]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings ('ignore', category = ConvergenceWarning)

# Params
TOLERANCE = 0.00005

# NOTE: Dropping negative records worsens results
train_data = pd.read_csv (TRAIN_PATH)
X_train = train_data.drop (TARGET_FEATURE, axis = 1)
Y_train = train_data[TARGET_FEATURE]

kf = KFold (n_splits = 5, shuffle = True, random_state = SEED)
fold_models = []
fold_scalers = []
fold_iterations = []

# Train all folds
for fold_idx, (train_idx, val_idx) in enumerate (kf.split (X_train)):
    print (f"\nTraining fold {fold_idx + 1}/5...")
    
    # Split fold into train/validation
    X_fold_train = X_train.iloc[train_idx]
    y_fold_train = Y_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]
    y_fold_val = Y_train.iloc[val_idx]
    
    # Preprocess fold data
    X_fold_train_proc, X_fold_val_proc, fold_scaler = \
                                            preprocess (X_fold_train.copy (), 
                                                        X_fold_val.copy ())
    
    # Train until test RMSE stops improving meaningfully
    fold_model = clone (model_template)
    prev_fold_rmse = float ('inf')
    fold_val_rmses = []
    fold_val_rmse = 0
    fold_iters = 0
    
    while fold_val_rmse + TOLERANCE < prev_fold_rmse:
        if fold_val_rmse > 0:
            prev_fold_rmse = fold_val_rmse
        
        fold_model.fit (X_fold_train_proc, y_fold_train)
        
        y_fold_val_pred = fold_model.predict (X_fold_val_proc)
        fold_val_rmse = np.sqrt (mean_squared_error (y_fold_val, y_fold_val_pred))
        fold_val_rmses.append (fold_val_rmse)
        
        fold_iters += 10
        print(f"  Iteration {fold_iters}: Val RMSE = {fold_val_rmse:.5f}")
    
    print (f"  Optimal iterations for fold {fold_idx + 1}: {fold_iters}")
    fold_iterations.append (fold_iters)
    fold_models.append (fold_model)
    fold_scalers.append (fold_scaler)

print (f"\nAverage final RMSEs: {sum (fold_val_rmses) / len (fold_val_rmses)}")


Training fold 1/5...
  Iteration 10: Val RMSE = 4.19313
  Iteration 20: Val RMSE = 4.14855
  Iteration 30: Val RMSE = 4.13910
  Iteration 40: Val RMSE = 4.13552
  Iteration 50: Val RMSE = 4.13378
  Iteration 60: Val RMSE = 4.13285
  Iteration 70: Val RMSE = 4.13234
  Iteration 80: Val RMSE = 4.13206
  Iteration 90: Val RMSE = 4.13191
  Iteration 100: Val RMSE = 4.13186
  Iteration 110: Val RMSE = 4.13187
  Optimal iterations for fold 1: 110

Training fold 2/5...
  Iteration 10: Val RMSE = 4.16399
  Iteration 20: Val RMSE = 4.12619
  Iteration 30: Val RMSE = 4.11841
  Iteration 40: Val RMSE = 4.11523
  Iteration 50: Val RMSE = 4.11358
  Iteration 60: Val RMSE = 4.11269
  Iteration 70: Val RMSE = 4.11220
  Iteration 80: Val RMSE = 4.11194
  Iteration 90: Val RMSE = 4.11181
  Iteration 100: Val RMSE = 4.11178
  Optimal iterations for fold 2: 100

Training fold 3/5...
  Iteration 10: Val RMSE = 4.16995
  Iteration 20: Val RMSE = 4.12965
  Iteration 30: Val RMSE = 4.12019
  Iteration 40: V

# Final Model

In [7]:
# Ensemble models
X_test = pd.read_csv (TEST_PATH)

# Ensemble predictions
print ("\nGenerating ensemble predictions...")
ensemble_preds = []

for fold_model, fold_scaler in zip (fold_models, fold_scalers):
    _, X_test_proc, _ = preprocess (X_train.copy (), 
                                    X_test.copy(), 
                                    scaler = fold_scaler)
    fold_pred = fold_model.predict (X_test_proc)
    ensemble_preds.append (fold_pred)

y_pred_ensemble = np.mean (ensemble_preds, axis = 0)

print (f"Ensemble pred mean: {y_pred_ensemble.mean ():.5f}")
print (f"Training label mean: {Y_train.mean ():.5f}")


out_data = pd.DataFrame ({'Cattle_ID': np.arange (1, len (y_pred_ensemble) + 1),
                          'Milk_Yield_L': y_pred_ensemble})
out_data.to_csv (OUT_PATH, index = False)


Generating ensemble predictions...
Ensemble pred mean: 15.63988
Training label mean: 15.58916
